# Objective 2: Resume Information Extraction Technique Comparison

This notebook tests different resume skill extraction techniques using the same resume input. The aim is to identify a suitable method for extracting skills, education, and experience before the extracted information is used in job matching, skill gap analysis, recommendation, and career path suggestion.

The techniques tested are:

1. Rule-based / keyword-based extraction
2. TF-IDF keyword extraction
3. SBERT similarity-based skill detection

NER is not implemented practically because it requires labelled resume or skill-span data for training and evaluation. This limitation is discussed in the report literature review.

In [2]:
# Install only if needed
# !pip install pdfplumber scikit-learn sentence-transformers pandas numpy

## 1. Import Libraries

In [4]:
import os
import re
import math
import numpy as np
import pandas as pd

try:
    import pdfplumber
except ImportError:
    raise ImportError("Please install pdfplumber first: pip install pdfplumber")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load Resume

Place the resume PDF in the same folder as this notebook. The default file name used here is `resume_hk.pdf`. If the file name is different, update the path below.

In [6]:
RESUME_PATH = "real resume/resume_hk.pdf"

# fallback for this notebook copy
if not os.path.exists(RESUME_PATH) and os.path.exists("/mnt/data/resume_hk(3).pdf"):
    RESUME_PATH = "/mnt/data/resume_hk(3).pdf"

print("Resume file:", RESUME_PATH)

Resume file: real resume/resume_hk.pdf


In [7]:
def extract_text_from_pdf(pdf_path):
    text_parts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            text_parts.append(page_text)
    return chr(10).join(text_parts)

resume_text = extract_text_from_pdf(RESUME_PATH)
print(resume_text[:2000])


HARIKUMAR MALAIRAJU
Petaling jaya, Selangor • 012-9980058 • hari040206@gmail.com
Linkedin: https://www.linkedin.com/in/harikumarmalairaju/
Jobstreet: https://my.jobstreet.com/profile/harikumar-malairaju-MBVMchLT7X
SUMMARY
Aspiring Data Scientist and second-year Computer Science (Hons.) student with developing skills in data
analysis, programming, and problem-solving. Demonstrated experience in managing data with high accuracy
and streamlining processes. Seeking a three-month data-focused internship from August 4 to October 26,
2025 to apply academic knowledge in a professional environment and contribute to data-driven decision-
making.
WORK EXPERIENCE
Part-Time Data Entry Employee, SV Design Solution Jan 2023 - Mar 2025
Maintained detailed financial logs and processed invoices in Excel.
Assisted in preparing accurate reports for audits and budget reviews.
Part-Time Data Entry Employee, Malai Sculptors and Builders Apr 2022 - Mar 2025
Accurately entered and maintained financial records,

## 3. Manual Ground Truth from Resume

The expected skills below are manually taken from the resume. This is used as the reference answer for validation.

In [9]:
expected_skills = [
    "python", "java", "c++", "r", "html", "css",
    "sql", "mysql", "ibm db2", "microsoft excel", "microsoft office suite",
    "project management", "problem solving", "analytical thinking",
    "time management", "communication", "adaptability", "teamwork"
]

expected_skills = sorted(set(expected_skills))
print("Total expected skills:", len(expected_skills))
expected_skills

Total expected skills: 18


['adaptability',
 'analytical thinking',
 'c++',
 'communication',
 'css',
 'html',
 'ibm db2',
 'java',
 'microsoft excel',
 'microsoft office suite',
 'mysql',
 'problem solving',
 'project management',
 'python',
 'r',
 'sql',
 'teamwork',
 'time management']

## 4. Skill Dictionary

The same type of skill dictionary is used to check technical and soft skills from resume text. This keeps the extraction output controlled and easy to explain.

In [11]:
skill_dictionary = [
    # programming and data
    "python", "java", "c++", "c#", "c", "r", "php", "javascript", "typescript",
    "html", "css", "sql", "mysql", "ibm db2", "oracle", "postgresql", "ms sql",
    "data analysis", "data science", "machine learning", "deep learning",
    "natural language processing", "nlp", "power bi", "tableau", "excel", "microsoft excel",
    "microsoft office", "microsoft office suite",
    
    # web and software
    "bootstrap", "react", "node.js", "django", "laravel", "spring", "docker",
    "kubernetes", "linux", "bash", "shell", "agile", "scrum", "devops",
    
    # soft skills
    "project management", "problem solving", "analytical thinking", "time management",
    "communication", "adaptability", "teamwork", "leadership", "customer service",
    "reporting", "accounting"
]

skill_dictionary = sorted(set([s.lower().strip() for s in skill_dictionary]))
print("Total skills in dictionary:", len(skill_dictionary))

Total skills in dictionary: 54


## 5. Helper Functions

In [13]:
def clean_text(text):
    text = text.lower()
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_skill(skill):
    return clean_text(skill)

def skill_pattern(skill):
    escaped = re.escape(skill)
    escaped = escaped.replace("\ ", r"\s+")
    return r"(?<![a-z0-9])" + escaped + r"(?![a-z0-9])"

def evaluate_extraction(method_name, extracted_skills, expected_skills):
    extracted = set(normalize_skill(s) for s in extracted_skills)
    expected = set(normalize_skill(s) for s in expected_skills)
    
    correct = sorted(extracted & expected)
    missed = sorted(expected - extracted)
    noisy = sorted(extracted - expected)
    
    precision = len(correct) / len(extracted) if extracted else 0
    recall = len(correct) / len(expected) if expected else 0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0
    
    return {
        "Method": method_name,
        "Extracted Count": len(extracted),
        "Correct Count": len(correct),
        "Missed Count": len(missed),
        "Noisy Count": len(noisy),
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "F1-score": round(f1, 3),
        "Correct Skills": ", ".join(correct),
        "Missed Skills": ", ".join(missed),
        "Noisy Skills": ", ".join(noisy)
    }

resume_clean = clean_text(resume_text)

## 6. Method 1: Rule-Based / Keyword-Based Extraction

This method checks whether known skill terms appear clearly in the resume text. It is simple, stable, and easy to explain.

In [15]:
def rule_based_extraction(text, skill_list):
    text = clean_text(text)
    detected = []
    for skill in skill_list:
        if re.search(skill_pattern(skill), text):
            detected.append(skill)
    return sorted(set(detected))

rule_based_skills = rule_based_extraction(resume_text, skill_dictionary)
print("Rule-based extracted skills:", len(rule_based_skills))
rule_based_skills

Rule-based extracted skills: 22


['adaptability',
 'analytical thinking',
 'c',
 'c++',
 'communication',
 'css',
 'data analysis',
 'data science',
 'excel',
 'html',
 'ibm db2',
 'java',
 'microsoft excel',
 'microsoft office',
 'microsoft office suite',
 'mysql',
 'project management',
 'python',
 'r',
 'sql',
 'teamwork',
 'time management']

## 7. Method 2: TF-IDF Keyword Extraction

This method extracts important keywords and phrases from the resume using TF-IDF. The extracted terms are then checked against the skill dictionary.

In [17]:
def tfidf_keyword_extraction(text, skill_list, top_n=35):
    lines = [line.strip() for line in text.splitlines() if len(line.strip()) > 2]
    if not lines:
        lines = [text]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 3),
        stop_words="english",
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9+#.]*\b"
    )
    tfidf_matrix = vectorizer.fit_transform(lines)
    feature_names = np.array(vectorizer.get_feature_names_out())
    scores = np.asarray(tfidf_matrix.sum(axis=0)).ravel()
    top_indices = scores.argsort()[::-1][:top_n]
    top_terms = [feature_names[i].lower().strip() for i in top_indices]

    detected = []
    for skill in skill_list:
        if skill in top_terms:
            detected.append(skill)
    return sorted(set(detected)), top_terms

tfidf_skills, tfidf_top_terms = tfidf_keyword_extraction(resume_text, skill_dictionary)

print("TF-IDF extracted skills:", len(tfidf_skills))
print("Top TF-IDF terms:")
print(tfidf_top_terms)
tfidf_skills


TF-IDF extracted skills: 8
Top TF-IDF terms:
['cgpa', 'skills', 'summary', 'management', 'excel', 'data', 'education', 'making', 'certifications', 'teamwork', 'communication', 'adaptability', 'aug', 'time', 'multimedia university', 'multimedia', 'cyberjaya', 'multimedia university cyberjaya', 'university', 'university cyberjaya', 'harikumar', 'harikumar malairaju', 'malairaju', 'problem solving', 'problem', 'solving', 'science', 'academic', 'experience', 'time management', 'microsoft', 'skills summary', 'project management', 'project', 'css']


['adaptability',
 'communication',
 'css',
 'excel',
 'problem solving',
 'project management',
 'teamwork',
 'time management']

## 8. Method 3: SBERT Similarity-Based Skill Detection

This method compares skill labels with the resume text using sentence embeddings. It can detect semantic similarity, but it may also return related skills that are not clearly written in the resume.

In [19]:
def sbert_similarity_extraction(text, skill_list, threshold=0.45):
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        print("sentence-transformers is not installed. Run: pip install sentence-transformers")
        return [], pd.DataFrame()

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    lines = [line.strip() for line in text.splitlines() if len(line.strip()) > 4]
    if not lines:
        lines = [text]

    skill_embeddings = model.encode(skill_list, convert_to_numpy=True, normalize_embeddings=True)
    line_embeddings = model.encode(lines, convert_to_numpy=True, normalize_embeddings=True)
    sim_matrix = cosine_similarity(skill_embeddings, line_embeddings)
    best_scores = sim_matrix.max(axis=1)

    rows = []
    detected = []
    for skill, score in zip(skill_list, best_scores):
        rows.append({"Skill": skill, "Best Similarity": round(float(score), 3)})
        if score >= threshold:
            detected.append(skill)

    score_df = pd.DataFrame(rows).sort_values("Best Similarity", ascending=False)
    return sorted(set(detected)), score_df

sbert_skills, sbert_scores = sbert_similarity_extraction(resume_text, skill_dictionary, threshold=0.45)
print("SBERT extracted skills:", len(sbert_skills))
sbert_skills


SBERT extracted skills: 25


['accounting',
 'adaptability',
 'analytical thinking',
 'c++',
 'communication',
 'css',
 'data analysis',
 'data science',
 'excel',
 'html',
 'ibm db2',
 'leadership',
 'microsoft excel',
 'microsoft office',
 'microsoft office suite',
 'ms sql',
 'mysql',
 'problem solving',
 'project management',
 'python',
 'reporting',
 'scrum',
 'sql',
 'teamwork',
 'time management']

In [20]:
if len(sbert_scores) > 0:
    display(sbert_scores.head(25))

,Skill,Best Similarity
1,adaptability,1.000
3,analytical thinking,1.000
9,communication,1.000
41,project management,1.000
51,teamwork,1.000
52,time management,1.000
40,problem solving,0.975
18,excel,0.807
20,ibm db2,0.792
28,microsoft excel,0.738


## 9. Validation Comparison

Each method is compared with the manual ground truth skills from the resume. The comparison uses precision, recall, F1-score, missed skills, and noisy skills.

In [22]:
results = []
results.append(evaluate_extraction("Rule-based / Keyword-based", rule_based_skills, expected_skills))
results.append(evaluate_extraction("TF-IDF Keyword Extraction", tfidf_skills, expected_skills))
results.append(evaluate_extraction("SBERT Similarity Extraction", sbert_skills, expected_skills))

comparison_df = pd.DataFrame(results)
summary_cols = [
    "Method", "Extracted Count", "Correct Count", "Missed Count", "Noisy Count",
    "Precision", "Recall", "F1-score"
]
comparison_df[summary_cols]

,Method,Extracted Count,Correct Count,Missed Count,Noisy Count,Precision,Recall,F1-score
0,Rule-based / Keyword-based,22,17,1,5,0.773,0.944,0.850
1,TF-IDF Keyword Extraction,8,7,11,1,0.875,0.389,0.538
2,SBERT Similarity Extraction,25,16,2,9,0.640,0.889,0.744


In [23]:
# Add suitability score for final prototype selection
# F1-score checks accuracy, while explainability checks how easy the output is to justify.
explainability_scores = {
    "Rule-based / Keyword-based": 1.00,
    "TF-IDF Keyword Extraction": 0.70,
    "SBERT Similarity Extraction": 0.60
}

selection_df = comparison_df[summary_cols].copy()
selection_df["Explainability"] = selection_df["Method"].map(explainability_scores)
selection_df["Prototype Suitability"] = (
    0.70 * selection_df["F1-score"] + 0.30 * selection_df["Explainability"]
).round(3)

selection_df = selection_df.sort_values(
    ["Prototype Suitability", "F1-score", "Noisy Count"],
    ascending=[False, False, True]
).reset_index(drop=True)

selection_df

,Method,Extracted Count,Correct Count,Missed Count,Noisy Count,Precision,Recall,F1-score,Explainability,Prototype Suitability
0,Rule-based / Keyword-based,22,17,1,5,0.773,0.944,0.850,1.0,0.895
1,SBERT Similarity Extraction,25,16,2,9,0.640,0.889,0.744,0.6,0.701
2,TF-IDF Keyword Extraction,8,7,11,1,0.875,0.389,0.538,0.7,0.587


## 10. Detailed Output by Method

In [25]:
for row in results:
    print()
    print(row["Method"])
    print("Correct:", row["Correct Skills"] if row["Correct Skills"] else "-")
    print("Missed :", row["Missed Skills"] if row["Missed Skills"] else "-")
    print("Noisy  :", row["Noisy Skills"] if row["Noisy Skills"] else "-")



Rule-based / Keyword-based
Correct: adaptability, analytical thinking, c++, communication, css, html, ibm db2, java, microsoft excel, microsoft office suite, mysql, project management, python, r, sql, teamwork, time management
Missed : problem solving
Noisy  : c, data analysis, data science, excel, microsoft office

TF-IDF Keyword Extraction
Correct: adaptability, communication, css, problem solving, project management, teamwork, time management
Missed : analytical thinking, c++, html, ibm db2, java, microsoft excel, microsoft office suite, mysql, python, r, sql
Noisy  : excel

SBERT Similarity Extraction
Correct: adaptability, analytical thinking, c++, communication, css, html, ibm db2, microsoft excel, microsoft office suite, mysql, problem solving, project management, python, sql, teamwork, time management
Missed : java, r
Noisy  : accounting, data analysis, data science, excel, leadership, microsoft office, ms sql, reporting, scrum


## 11. Final Selected Method

The selected method is based on extraction performance and suitability for the final prototype.

In [27]:
selected_method = selection_df.iloc[0]["Method"]
print("Selected extraction method:", selected_method)

if selected_method == "Rule-based / Keyword-based":
    final_extracted_skills = rule_based_skills
elif selected_method == "TF-IDF Keyword Extraction":
    final_extracted_skills = tfidf_skills
else:
    final_extracted_skills = sbert_skills

print("Final extracted skills used for the prototype:")
final_extracted_skills

Selected extraction method: Rule-based / Keyword-based
Final extracted skills used for the prototype:


['adaptability',
 'analytical thinking',
 'c',
 'c++',
 'communication',
 'css',
 'data analysis',
 'data science',
 'excel',
 'html',
 'ibm db2',
 'java',
 'microsoft excel',
 'microsoft office',
 'microsoft office suite',
 'mysql',
 'project management',
 'python',
 'r',
 'sql',
 'teamwork',
 'time management']

## 12. Education and Experience Extraction

The final dashboard also extracts education and experience information. This section checks whether the rule-based profile extraction can identify basic education and experience details from the resume.

In [29]:
def extract_education(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    education_keywords = ["bachelor", "foundation", "diploma", "degree", "university", "college", "cgpa"]
    education_lines = []
    for line in lines:
        low = line.lower()
        if any(k in low for k in education_keywords):
            education_lines.append(line)
    return education_lines

def extract_experience_years(text):
    text_low = text.lower()
    ranges = re.findall(r"(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\s+\d{4}\s*-\s*(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\s+\d{4}", text_low)
    # This simple check confirms that work date ranges are detected.
    return len(ranges)

education_output = extract_education(resume_text)
experience_ranges_detected = extract_experience_years(resume_text)

print("Education lines detected:")
for item in education_output:
    print("-", item)

print()
print("Work date ranges detected:", experience_ranges_detected)


Education lines detected:
- Bachelor of Computer Science (Undergraduate) Aug 2023 - Aug 2026
- Multimedia University, Cyberjaya
- CGPA: 3.43
- Foundation in Information Technology Aug 2022 - Aug 2023
- Multimedia University, Cyberjaya
- CGPA: 2.99

Work date ranges detected: 4


## 13. Conclusion for Objective 2

Based on the validation using the real resume, the rule-based / keyword-based extraction method is selected for the final prototype. It gives controlled output, lower noise, and clear explainability. TF-IDF is useful as a baseline keyword method, but it depends on term importance and may miss skills that are not ranked highly. SBERT can support semantic matching, but it may return related skills that are not directly stated in the resume.

Therefore, the final Streamlit dashboard continues to use the rule-based extraction method for resume skills, education, and experience extraction.